# Chapter 19 — The Full Training Loop on GPU

> Course: **llm.c — Zero to Hero**, Chapter 19 of ~20.
> Builds on every prior chapter.

This is the payoff chapter. You've learned every piece. Now we'll trace `train_gpt2.cu` from `main()` to the final training step and see how all the kernels you've studied — `encoder_forward_kernel3`, `layernorm_forward_kernel6`, `matmul_cublaslt`, `attention_forward_cudnn`, `gelu_forward_kernel2`, `fused_residual_forward_kernel`, `fused_classifier_kernel`, `adamw_kernel3`, `global_norm_squared_kernel` — fit together into ~250 lines of orchestration code.

This chapter is mostly **walkthrough**. The runnable bit is building the actual `train_gpt2cu` binary and (optionally) running it on the starter pack to confirm everything we've built reproduces real GPT-2 training.

### Learning objectives

By the end of this chapter you will:

- Trace `gpt2_forward` in `train_gpt2.cu` from inputs → loss, naming every kernel call.
- Trace `gpt2_backward_and_reduce` in reverse, naming every kernel call.
- Understand **gradient accumulation** and how the optimizer step is gated on it.
- Be able to compile and (optionally) run the full GPU GPT-2.


## 1. The `gpt2_forward` Walk in `train_gpt2.cu`

The CUDA forward (lines 646-755 in train_gpt2.cu) mirrors the CPU forward (Chapter 8) almost step for step. The differences are *which* kernel is called for each layer:

| Step | CPU `train_gpt2.c` | GPU `train_gpt2.cu` |
|---|---|---|
| Embedding | `encoder_forward` | `encoder_forward(...)` → `encoder_forward_kernel3` |
| LayerNorm | `layernorm_forward` | `layernorm_forward(...)` → `layernorm_forward_kernel6` (warp-per-row + Packed128) |
| QKV linear | `matmul_forward` | `matmul_forward_cublaslt` (cuBLAS!) |
| QKV permute | n/a (packed used directly) | `permute_kernel` (Chapter 16) |
| Attention | `attention_forward` | `attention_forward(...)` — either `softmax_forward_kernel5` + cuBLAS QK·KᵀV, or **`attention_forward_cudnn`** Flash Attention |
| Attention output unpermute | n/a | `unpermute_kernel` |
| Attention output linear | `matmul_forward` | `matmul_forward_cublaslt` |
| Residual + LayerNorm fused | two calls | **`fused_residual_forward_kernel`** (Chapter 15) |
| FFN up + GELU fused | two calls | `matmul_forward_cublaslt` with `EPILOGUE_GELU_AUX_BIAS` (Chapter 14) |
| FFN down | `matmul_forward` | `matmul_forward_cublaslt` |
| Final residual + LN + matmul to logits | three calls | last block: fused residual+LN, then matmul |
| Loss head | `softmax_forward` + `crossentropy_forward` | `fused_classifier_kernel` (Chapter 7+15) — softmax+CE+backward all in one |

Notable: the **last block fuses residual+LN** (saves a memory pass), and the **classifier fuses softmax + CE_forward + dlogits** (avoids materializing the giant `(B, T, Vp)` probs tensor). Both fusions you saw in Chapter 15.

The actual CUDA forward function:

```cpp
void gpt2_forward(GPT2 *model, const int* inputs, size_t B, size_t T, int grad_accum_steps=1) {
    // ... lazy alloc activations ...
    // ... validate inputs ...

    encoder_forward(acts.encoded, model->inputs, params.wte, params.wpe, B, T, C);

    for (int l = 0; l < L; l++) {
        residual = (l == 0) ? acts.encoded : acts.residual3 + (l-1)*B*T*C;
        // pull per-layer pointers...

        layernorm_forward(l_ln1, l_ln1_mean, l_ln1_rstd, residual, l_ln1w, l_ln1b, B, T, C);
        matmul_forward_cublaslt(l_qkvr, l_ln1, l_qkvw, l_qkvb, B, T, C, 3*C);
        attention_forward(l_atty, l_qkvr, l_att, B, T, C, NH);     // dispatches to cudnn or hand-rolled
        matmul_forward_cublaslt(l_residual2, l_atty, l_attprojw, l_attprojb, B, T, C, C,
                                stream, true, false, 0, 0, 0, 0, true, NULL, false);  // accumulate=true
        // ... fused residual + LN1 / LN2 ...
        matmul_forward_cublaslt(l_fch_gelu, l_ln2, l_fcw, l_fcb, B, T, C, 4*C,
                                stream, true, false, 0, 0, 0, 0, false, l_fch, false);  // GELU epilogue, save pre-gelu
        matmul_forward_cublaslt(scratch, l_fch_gelu, l_fcprojw, l_fcprojb, B, T, 4*C, C,
                                stream, true, false, 0, 0, 0, 0, true, NULL, false);   // accumulate
    }
    // final LN + classifier
    fused_residual_forward(... last residual + final LN ...);
    matmul_forward_cublaslt(acts.output, acts.lnf, params.wte, NULL, B, T, C, Vp);
    fused_classifier(acts.output, acts.losses, dloss_per_position, model->targets, B, T, V, Vp);
}
```

Notice the `accumulate=true` flag passed to some matmul calls — that's how `llm.c` *fuses* the residual addition into the output projection (matmul writes `C += A @ B + bias`, completing both the projection and the residual in one kernel).


## 2. The Backward — `gpt2_backward_and_reduce`

The backward function (lines 788-1169 in train_gpt2.cu) does **gradient accumulation, kernel orchestration in reverse, and (in multi-GPU) NCCL all-reduce** all in one function. The reverse-order layer calls match what you saw in Chapter 8:

```cpp
void gpt2_backward_and_reduce(GPT2 *model, ...) {
    // STEP 1: kick off chain rule from the loss
    fused_classifier(...);        // already produced dlogits in the forward!

    // STEP 2: matmul backward through the unembedding (uses params.wte → grads.wte)
    matmul_backward_cublaslt(grads_acts.lnf, grads.wte, NULL, grads_acts.logits, ...);

    // STEP 3: layernorm backward through the final LN
    layernorm_backward(grads_acts.lnf_input, grads.lnfw, grads.lnfb, ...);

    // STEP 4: per-layer backward, in REVERSE
    for (int l = L-1; l >= 0; l--) {
        matmul_backward_cublaslt(...);     // FFN down backward (with GELU backward fused via EPILOGUE_DGELU_BGRADB)
        matmul_backward_cublaslt(...);     // FFN up backward
        layernorm_backward(...);           // LN2 backward
        residual_backward(...);            // residual gradient split (Chapter 5)
        matmul_backward_cublaslt(...);     // attention output projection backward
        attention_backward(...);           // either cudnn or hand-rolled
        matmul_backward_cublaslt(...);     // QKV backward
        layernorm_backward(...);           // LN1 backward
        residual_backward(...);            // first residual gradient split
    }

    // STEP 5: encoder backward (writes to grads.wte and grads.wpe)
    encoder_backward(grads.wte, grads.wpe, grads_acts.encoded, model->inputs, B, T, C);

    // STEP 6 (multi-GPU only): all-reduce gradients across processes
    if (multi_gpu_config.num_processes > 1) {
        ncclAllReduce(grads_memory, grads_memory, num_parameters, ncclFloat, ncclSum, ...);
    }
}
```

Notice **`fused_classifier`** is called *only once* — it produces dlogits during the forward, so the backward starts right at the unembedding matmul. This is the subtle trick from Chapter 7: dlogits depends only on `(probs, target)`, both of which the forward classifier kernel can compute inline.


## 3. Concept — Gradient Accumulation

Real GPT-2 training uses a "global batch" larger than what fits on one GPU. The trick: do `K` forward+backward steps **without** running AdamW, accumulating gradients into the same `grads_memory` (each backward writes with `+=`). After `K` micro-batches, run AdamW *once* on the summed gradients.

In `train_gpt2.cu`'s training loop:

```cpp
for (int step = 1; step <= num_iterations; step++) {
    gpt2_zero_grad(model);

    for (int grad_step = 0; grad_step < grad_accum_steps; grad_step++) {
        dataloader_next_batch(loader);
        gpt2_forward(model, loader->inputs, loader->targets, B, T, grad_accum_steps);
        gpt2_backward_and_reduce(model, ...);
    }

    // After grad_accum_steps mini-forwards/backwards, do ONE AdamW step
    gpt2_update(model, lr, beta1, beta2, eps, wd, step, max_grad_norm);
}
```

The `grad_accum_steps` parameter is also passed into `fused_classifier` so it scales the loss by `1/grad_accum_steps` — that's the gradient-of-mean correction for averaging the loss across K micro-batches.

This is how 1B+ parameter models are trained on 8-GPU nodes: tile the global batch into many small chunks, accumulate, run optimizer once.


## 4. Compile the Real Thing

You can build `train_gpt2cu` right now:

```bash
make train_gpt2cu                  # default BF16 build, no cuDNN
make train_gpt2cu USE_CUDNN=1      # BF16 + cuDNN Flash Attention (faster)
make train_gpt2fp32cu              # FP32 build (slower but simpler debugging)
make train_gpt2cu NO_MULTI_GPU=1   # single-GPU box without NCCL installed
```

On a single-GPU machine that doesn't have NCCL installed, the default build fails on
`#include <nccl.h>` (pulled in by `llmc/zero.cuh`). Pass `NO_MULTI_GPU=1` to compile the
single-GPU path — that's what we do below, since this is a one-GPU box.

Let's compile it (without running) to confirm everything builds.


In [ ]:
# Quick build of train_gpt2cu — uses NVCC, takes ~30s
# NO_MULTI_GPU=1 builds the single-GPU path (skips NCCL, which isn't installed here).
import subprocess
r = subprocess.run(["make", "train_gpt2cu", "NO_MULTI_GPU=1"], capture_output=True, text=True)
print(r.stdout[-500:] if len(r.stdout) > 500 else r.stdout)
if r.returncode != 0:
    print("STDERR:")
    print(r.stderr[-1500:])
else:
    print("BUILD OK")
    # check the binary exists
    import os
    if os.path.exists("train_gpt2cu"):
        print(f"  binary size: {os.path.getsize('train_gpt2cu')/1024:.1f} KB")


If the build succeeded, you have `train_gpt2cu` ready to run. To actually train:

```bash
./dev/download_starter_pack.sh        # ~520 MB — GPT-2 124M weights, tokenizer, debug state, dataset
./train_gpt2cu                         # train from the loaded checkpoint
```

The starter pack download is a one-time ~520 MB. Once you have it, `./train_gpt2cu` will train GPT-2 on tinyshakespeare and print loss going down. Every kernel it calls is one you've now studied.

This is **optional** — completing the course doesn't require running it.


## 5. Validate Against PyTorch — the `test_gpt2cu` Binary

`test_gpt2.cu` (in the repo root) loads the same `gpt2_124M_debug_state.bin` from `train_gpt2.py`, runs forward and backward, and compares **every intermediate tensor** plus losses and gradients against PyTorch's reference values bit-by-bit.

```bash
make test_gpt2cu                     # build the test
./test_gpt2cu                         # runs forward, backward, 10 AdamW steps, checks everything
```

The output is a dozen lines like:

```
[State]
OK (LOGITS): maxdiff = 5.341... (tol 1.000000e-02)
LOSS OK: 5.270 5.270
dwte:        TENSOR OK, maxdiff = 1.86e-04
dwpe:        TENSOR OK, maxdiff = 6.10e-05
... etc ...
step 0: loss 5.270 (took ... ms)
step 1: loss 4.063 ...
...
```

If you see all `OK` and `loss` going down, **your understanding of `llm.c` is complete**: every line of code you've read produces the same numbers as PyTorch.


## 6. The Mental Map

The full GPU stack, ordered from outermost loop in:

```
main()                                         (train_gpt2.cu)
└─ training loop
   ├─ dataloader_next_batch                    (llmc/dataloader.h)
   ├─ for grad_step:
   │  ├─ gpt2_forward
   │  │  ├─ encoder_forward                    (llmc/encoder.cuh — Ch 10)
   │  │  ├─ for layer:
   │  │  │  ├─ layernorm_forward               (llmc/layernorm.cuh — Ch 13)
   │  │  │  ├─ matmul_forward_cublaslt × N     (llmc/matmul.cuh — Ch 14)
   │  │  │  ├─ attention_forward               (llmc/attention.cuh or cudnn_att.h — Ch 16)
   │  │  │  └─ fused_residual_forward          (Ch 15)
   │  │  └─ fused_classifier (forward+dlogits) (llmc/fused_classifier.cuh — Chs 7, 15)
   │  └─ gpt2_backward_and_reduce
   │     ├─ matmul_backward_cublaslt × N       (Ch 14)
   │     ├─ layernorm_backward                 (Ch 13)
   │     ├─ attention_backward                 (Ch 16)
   │     ├─ encoder_backward                   (Ch 10)
   │     └─ NCCL all-reduce (multi-GPU)        (Ch 20)
   ├─ global_norm + clip                       (llmc/global_norm.cuh — Ch 18)
   ├─ adamw_kernel3                            (llmc/adamw.cuh — Ch 18)
   └─ copy_and_cast (FP32 master → BF16)       (Ch 17)
```

Every `Ch X` reference points back to a chapter where you've already seen — and probably written — that piece.


## Recap

You now know:

- `gpt2_forward` in `train_gpt2.cu` is **the same orchestration as the CPU version**, with every layer call routed to a CUDA kernel (cuBLAS or hand-rolled).
- The classifier fuses **softmax + CE + dlogits** in the forward — saving a giant probs tensor and avoiding a separate "backward through softmax" kernel.
- **Gradient accumulation** lets you train on global batches bigger than one GPU's memory: K micro-batches add into the same `grads_memory`, then one AdamW step.
- `test_gpt2cu` cross-checks every intermediate tensor against PyTorch — your guarantee that the code is correct.

### What's next

**Chapter 20 — Multi-GPU with NCCL & ZeRO.** The final chapter. We'll see how `llmc/zero.cuh` shards the optimizer state across GPUs (ZeRO-1) and uses NCCL to all-reduce gradients between processes. Same gradient pipeline, but now spread across 8 GPUs at training time.

When you're ready, say **"proceed to Chapter 20"**.
